### Attribution
https://github.com/miptgirl/miptgirl_medium/blob/main/dspy_example/nps_topic_modelling.ipynb

In [1]:
import pandas as pd
import tqdm
import dspy
from utils import wrap_text

In [2]:

SMALL_MODEL_CANDIDATES: list[str] = [
    "gemini/gemini-2.5-flash-lite",
    "gemini/gemini-2.5-flash",
    "gemini/gemini-2.0-flash",
]

REFLECTION_MODEL_CANDIDATES: list[str] = [
    "gemini/gemini-3.1-pro-preview",
    "gemini/gemini-2.5-pro",
    "gemini/gemini-1.5-pro",
]

llm = dspy.LM(SMALL_MODEL_CANDIDATES[0])
dspy.configure(lm=llm)
dspy.configure_cache(enable_memory_cache=False, enable_disk_cache=False)

# Net Promoter Score 
Net Promoter Score (NPS) is a customer loyalty metric that measures the likelihood of customers recommending a company, ranging from -100 to 100. It is calculated by subtracting the percentage of detractors (0–6 score) from promoters (9–10 score). A score above 0 is good, 50+ is excellent, and 70+ is world-class.

That said, we're using the 'topics' and 'comment' fields to build a classifier 


In [3]:
import json
with open('nps_comments.json', 'r') as f:
    nps_data = json.loads(f.read())
print(f'total samples: {len(nps_data)}')
print(f"sample:\n {nps_data[0]}")

total samples: 105
sample:
 {'topics': ['Limited Size or Shade Availability'], 'comment': "Absolutely frustrated! Every time I find something I love, it's sold out in my size. What's the point of having a wishlist if nothing is ever available?"}


#### Get all topics

In [4]:
topics = set()

for rec in nps_data: 
    for t in rec['topics']: 
        topics.add(t)

In [5]:
topic_list = list(topics)
print(wrap_text(str(topic_list)))
print(f"\nTopic count: {len(topic_list)}")

['Unresponsive or Generic Customer Support', 'Difficult Product Discovery',
'Website or App Bugs', 'Customs and Import Charges', 'Confusing Loyalty or
Discount Systems', 'Limited Size or Shade Availability', 'Slow or Unreliable
Shipping', 'Inaccurate Product Descriptions or Photos', 'Complicated Returns or
Exchanges', 'Damaged or Incorrect Items']

Topic count: 10


#### Prompt Optimization with MIPROv2

In [6]:
from typing import Literal, List

class NPSTopic(dspy.Signature):
    """Classify NPS topics"""

    comment: str = dspy.InputField()
    answer: List[Literal[*topic_list]] = dspy.OutputField()

In [7]:
print(nps_data[0]['topics'])
print(wrap_text(nps_data[0]['comment'], width=72))

['Limited Size or Shade Availability']
Absolutely frustrated! Every time I find something I love, it's sold out
in my size. What's the point of having a wishlist if nothing is ever
available?


In [8]:
nps_topic_precictor_w_reasoning = dspy.ChainOfThought(NPSTopic)  # dspy CoT adds a reasoning field to the prediction
response = nps_topic_precictor_w_reasoning(comment = "Absolutely frustrated! Every time I find something I love, it's sold out in my size. What's the point of having a wishlist if nothing is ever available?")
print(response)
print(response.reasoning)
print(response.answer)


Prediction(
    reasoning='The user is expressing frustration because items they are interested in are frequently unavailable in their size. This directly points to a problem with the availability of products in different sizes.',
    answer=['Limited Size or Shade Availability']
)
The user is expressing frustration because items they are interested in are frequently unavailable in their size. This directly points to a problem with the availability of products in different sizes.
['Limited Size or Shade Availability']


In [9]:
dspy.inspect_history(n = 1)






[2026-05-13T22:00:54.825916]

System message:

Your input fields are:
1. `comment` (str):
Your output fields are:
1. `reasoning` (str): 
2. `answer` (list[Literal['Unresponsive or Generic Customer Support', 'Difficult Product Discovery', 'Website or App Bugs', 'Customs and Import Charges', 'Confusing Loyalty or Discount Systems', 'Limited Size or Shade Availability', 'Slow or Unreliable Shipping', 'Inaccurate Product Descriptions or Photos', 'Complicated Returns or Exchanges', 'Damaged or Incorrect Items']]):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## comment ## ]]
{comment}

[[ ## reasoning ## ]]
{reasoning}

[[ ## answer ## ]]
{answer}        # note: the value you produce must adhere to the JSON schema: {"type": "array", "items": {"type": "string", "enum": ["Unresponsive or Generic Customer Support", "Difficult Product Discovery", "Website or App Bugs", "Customs and Import Charges", "Confusing Loyalty or Discount System

In [10]:
nps_df = pd.DataFrame(nps_data)
# Net effect: each row gets a 1-based id (1, 2, 3, …, N) in a new column.
nps_df['id'] = list(map(lambda x: x + 1, range(nps_df.shape[0])))

In [11]:
tmp = []

for rec in tqdm.tqdm(nps_df.to_dict('records')):
    response = nps_topic_precictor_w_reasoning(comment = rec['comment'])  # dspy.InputField comment
    res = {
        'id': rec['id'],
        'predicted_topics': response.answer
    }

    tmp.append(res)

100%|██████████| 105/105 [02:32<00:00,  1.45s/it]


In [12]:
ini_model_topics_df = pd.DataFrame(tmp)

In [13]:
ini_model_topics_df

,id,predicted_topics
0,1,[Limited Size or Shade Availability]
1,2,"[Difficult Product Discovery, Website or App B..."
2,3,[Inaccurate Product Descriptions or Photos]
3,4,"[Confusing Loyalty or Discount Systems, Unresp..."
4,5,[Website or App Bugs]
...,...,...
100,101,"[Customs and Import Charges, Inaccurate Produc..."
101,102,[Confusing Loyalty or Discount Systems]
102,103,"[Website or App Bugs, Unresponsive or Generic ..."
103,104,"[Difficult Product Discovery, Inaccurate Produ..."


In [14]:
nps_df = nps_df.merge(ini_model_topics_df)

In [15]:
# show head
nps_df.sample(5).to_dict('records')

[{'topics': ['Limited Size or Shade Availability'],
  'comment': "Travel sizes sell out faster than full sizes but aren't restocked as frequently. Poor inventory balance.",
  'id': 81,
  'predicted_topics': ['Limited Size or Shade Availability']},
 {'topics': ['Inaccurate Product Descriptions or Photos'],
  'comment': 'Serum texture was completely different from description. Said lightweight, received thick cream. Very misleading.',
  'id': 39,
  'predicted_topics': ['Inaccurate Product Descriptions or Photos']},
 {'topics': ['Website or App Bugs',
   'Unresponsive or Generic Customer Support'],
  'comment': "App keeps logging me out mid-checkout. Support says it's my phone's fault but it only happens on your app.",
  'id': 99,
  'predicted_topics': ['Website or App Bugs']},
 {'topics': ['Confusing Loyalty or Discount Systems'],
  'comment': 'Thought I qualified for free shipping with my membership but got charged anyway. Your terms and conditions are confusing and misleading.',
  'id'

In [16]:
def compare_GT_predicted_topics(l1, l2):
    l1_fmt = ', '.join(sorted(l1))
    l2_fmt = ', '.join(sorted(l2))
    if l1_fmt == l2_fmt: 
        return 1 
    return 0


nps_df['predicted_accuracy'] = list(map(
    compare_GT_predicted_topics,
    nps_df.topics,
    nps_df.predicted_topics
))

In [ ]:
# accuracy
round(100*nps_df.predicted_accuracy.mean(), 2)

np.float64(82.86)

# Optimizing the Prompt

In [18]:
import random
# create a training set and a validation set
trainset = []
valset = []
for rec in nps_data: 
    if random.random() <= 0.5:
        trainset.append(
            dspy.Example(
                comment = rec['comment'],
                answer = rec['topics']
            ).with_inputs('comment')
        )
    else: 
        valset.append(
            dspy.Example(
                comment = rec['comment'],
                answer = rec['topics']
            ).with_inputs('comment')
        )

In [19]:
# tp = dspy.MIPROv2(metric=dspy.evaluate.answer_exact_match, auto="light", num_threads=24)

In [20]:
def list_exact_match(example, pred, trace=None):
    """Custom metric for comparing lists of topics"""
    try:
        pred_answer = pred.answer
        expected_answer = example.answer
        
        # Convert to sets for order-independent comparison
        if isinstance(pred_answer, list) and isinstance(expected_answer, list):
            return set(pred_answer) == set(expected_answer)
        else:
            return pred_answer == expected_answer
    except Exception as e:
        print(f"Error in metric: {e}")
        return False

In [21]:
mipro_tele_prompter = dspy.MIPROv2(metric=list_exact_match, auto="light", num_threads=24)

In [22]:
mipro_optimized_nps_topic_predictor =  mipro_tele_prompter.compile(
    nps_topic_precictor_w_reasoning, 
    trainset=trainset, 
    valset=valset,
    requires_permission_to_run = False, provide_traceback=True)

2026/05/13 22:03:27 WARNING dspy.teleprompt.mipro_optimizer_v2: 'requires_permission_to_run' is deprecated and will be removed in a future version.
2026/05/13 22:03:27 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING LIGHT AUTO RUN SETTINGS:
num_trials: 10
minibatch: True
num_fewshot_candidates: 6
num_instruct_candidates: 3
valset size: 53

2026/05/13 22:03:27 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2026/05/13 22:03:27 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2026/05/13 22:03:27 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=6 sets of demonstrations...


Bootstrapping set 1/6
Bootstrapping set 2/6
Bootstrapping set 3/6


  8%|▊         | 4/52 [00:08<01:37,  2.04s/it]


Bootstrapped 4 full traces after 4 examples for up to 1 rounds, amounting to 4 attempts.
Bootstrapping set 4/6


  6%|▌         | 3/52 [00:05<01:23,  1.71s/it]


...

2026/05/13 22:05:24 INFO dspy.evaluate.evaluate: Average Metric: 30 / 35 (85.7%)
2026/05/13 22:05:24 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 85.71 on minibatch of size 35 with parameters ['Predictor 0: Instruction 0', 'Predictor 0: Few-Shot Set 1'].
2026/05/13 22:05:24 INFO dspy.teleprompt.mipro_optimizer_v2: Minibatch scores so far: [71.43, 77.14, 85.71, 74.29, 88.57, 74.29, 85.71, 71.43, 85.71]
2026/05/13 22:05:24 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [83.02, 84.91]
2026/05/13 22:05:24 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 84.91
2026/05/13 22:05:24 INFO dspy.teleprompt.mipro_optimizer_v2: ==========================================


2026/05/13 22:05:24 INFO dspy.teleprompt.mipro_optimizer_v2: == Trial 12 / 13 - Minibatch ==



Average Metric: 28.00 / 35 (80.0%): 100%|██████████| 35/35 [00:05<00:00,  6.25it/s]

2026/05/13 22:05:30 INFO dspy.evaluate.evaluate: Average Metric: 28 / 35 (80.0%)
2026/05/13 22:05:30 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 80.0 on minibatch of size 35 with parameters ['Predictor 0: Instruction 1', 'Predictor 0: Few-Shot Set 5'].
2026/05/13 22:05:30 INFO dspy.teleprompt.mipro_optimizer_v2: Minibatch scores so far: [71.43, 77.14, 85.71, 74.29, 88.57, 74.29, 85.71, 71.43, 85.71, 80.0]
2026/05/13 22:05:30 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [83.02, 84.91]
2026/05/13 22:05:30 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 84.91
2026/05/13 22:05:30 INFO dspy.teleprompt.mipro_optimizer_v2: ==========================================


2026/05/13 22:05:30 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Trial 13 / 13 - Full Evaluation =====
2026/05/13 22:05:30 INFO dspy.teleprompt.mipro_optimizer_v2: Doing full eval on next top averaging program (Avg Score: 85.71) from minibatch trials...



Average Metric: 43.00 / 53 (81.1%): 100%|██████████| 53/53 [00:05<00:00,  9.80it/s]

2026/05/13 22:05:35 INFO dspy.evaluate.evaluate: Average Metric: 43 / 53 (81.1%)
2026/05/13 22:05:35 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [83.02, 84.91, 81.13]
2026/05/13 22:05:35 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 84.91
2026/05/13 22:05:35 INFO dspy.teleprompt.mipro_optimizer_v2: =======================
2026/05/13 22:05:35 INFO dspy.teleprompt.mipro_optimizer_v2: 

2026/05/13 22:05:35 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 84.91!


In [23]:
mipro_optimized_nps_topic_predictor(
    comment = "Absolutely frustrated! Every time I find something I love, it's sold out in my size."
    "What's the point of having a wishlist if nothing is ever available?"
)

Prediction(
    reasoning='The user\'s comment explicitly states frustration because items are "sold out in my size" and questions the point of a wishlist if "nothing is ever available." This directly points to a lack of available sizes for the products the user is interested in.',
    answer=['Limited Size or Shade Availability']
)

In [24]:
dspy.inspect_history(n = 1)





[2026-05-13T22:05:36.776615]

System message:

Your input fields are:
1. `comment` (str):
Your output fields are:
1. `reasoning` (str): 
2. `answer` (list[Literal['Unresponsive or Generic Customer Support', 'Difficult Product Discovery', 'Website or App Bugs', 'Customs and Import Charges', 'Confusing Loyalty or Discount Systems', 'Limited Size or Shade Availability', 'Slow or Unreliable Shipping', 'Inaccurate Product Descriptions or Photos', 'Complicated Returns or Exchanges', 'Damaged or Incorrect Items']]):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## comment ## ]]
{comment}

[[ ## reasoning ## ]]
{reasoning}

[[ ## answer ## ]]
{answer}        # note: the value you produce must adhere to the JSON schema: {"type": "array", "items": {"type": "string", "enum": ["Unresponsive or Generic Customer Support", "Difficult Product Discovery", "Website or App Bugs", "Customs and Import Charges", "Confusing Loyalty or Discount System

In [25]:
# test on valset

tmp = []

for e in tqdm.tqdm(valset):
    comment = e.comment 
    baseline_resp = nps_topic_precictor_w_reasoning(comment = comment) # TODO, is this a repeat
    mipro_resp = mipro_optimized_nps_topic_predictor(comment = comment)

    tmp.append(
        {
        'comment': comment,
        'ground_truth_answer': e.answer,
        'baseline_answer': baseline_resp.answer,
        'mipro_answer': mipro_resp.answer
        }
    )

100%|██████████| 53/53 [02:17<00:00,  2.60s/it]


In [26]:
cmp_df = pd.DataFrame(tmp)

In [27]:
def list_exact_match_raw(expected_answer, pred_answer, trace=None):
    """Custom metric for comparing lists of topics"""
    try:
        # Convert to sets for order-independent comparison
        if isinstance(pred_answer, list) and isinstance(expected_answer, list):
            return set(pred_answer) == set(expected_answer)
        else:
            return pred_answer == expected_answer
    except Exception as e:
        print(f"Error in metric: {e}")
        return False

In [30]:
cmp_df['baseline_accuracy'] = list(map(
    list_exact_match_raw,
    cmp_df.ground_truth_answer,
    cmp_df.baseline_answer))

cmp_df['mipro_accuracy'] = list(map(
    list_exact_match_raw,
    cmp_df.ground_truth_answer,
    cmp_df.mipro_answer))

In [31]:
cmp_df[['baseline_accuracy', 'mipro_accuracy']].mean()*100

baseline_accuracy    79.245283
mipro_accuracy       84.905660
dtype: float64

### dspy.BootstrapFewShotWithRandomSearch

In [32]:
bfs_rs = dspy.BootstrapFewShotWithRandomSearch(list_exact_match, num_threads=24, max_bootstrapped_demos = 10)

Going to sample between 1 and 10 traces per predictor.
Will attempt to bootstrap 16 candidate sets.


In [34]:
bfs_rs_optimized_nps_topic_predictor =  bfs_rs.compile(
    nps_topic_precictor_w_reasoning, 
    trainset=trainset, 
    valset=valset)

Average Metric: 44.00 / 53 (83.0%): 100%|██████████| 53/53 [00:04<00:00, 12.79it/s] 

2026/05/13 22:14:43 INFO dspy.evaluate.evaluate: Average Metric: 44 / 53 (83.0%)



New best score: 83.02 for seed -3
Scores so far: [83.02]
Best score so far: 83.02
Average Metric: 41.00 / 53 (77.4%): 100%|██████████| 53/53 [00:03<00:00, 14.58it/s]

2026/05/13 22:14:46 INFO dspy.evaluate.evaluate: Average Metric: 41 / 53 (77.4%)



Scores so far: [83.02, 77.36]
Best score so far: 83.02


...


Scores so far: [83.02, 77.36, 77.36, 83.02, 77.36, 77.36, 77.36, 81.13, 83.02, 77.36, 77.36, 81.13, 77.36, 81.13, 83.02, 77.36, 77.36, 77.36]
Best score so far: 83.02


 10%|▉         | 5/52 [00:04<00:42,  1.12it/s]


Bootstrapped 4 full traces after 5 examples for up to 1 rounds, amounting to 5 attempts.
Average Metric: 41.00 / 53 (77.4%): 100%|██████████| 53/53 [00:06<00:00,  7.59it/s]

2026/05/13 22:17:50 INFO dspy.evaluate.evaluate: Average Metric: 41 / 53 (77.4%)



Scores so far: [83.02, 77.36, 77.36, 83.02, 77.36, 77.36, 77.36, 81.13, 83.02, 77.36, 77.36, 81.13, 77.36, 81.13, 83.02, 77.36, 77.36, 77.36, 77.36]
Best score so far: 83.02
19 candidate programs found.


In [35]:
tmp = []

for e in tqdm.tqdm(valset):
    comment = e.comment 
    baseline_resp = nps_topic_precictor_w_reasoning(comment = comment) 
    mipro_resp = mipro_optimized_nps_topic_predictor(comment = comment)
    bfs_rs_resp = bfs_rs_optimized_nps_topic_predictor(comment = comment)

    tmp.append(
        {
        'comment': comment,
        'gold_answer': e.answer,
        'baseline_answer': baseline_resp.answer,
        'mipro_answer': mipro_resp.answer,
        'bfs_rs_answer': bfs_rs_resp.answer
        }
    )

100%|██████████| 53/53 [03:23<00:00,  3.85s/it]


In [36]:
cmp_df = pd.DataFrame(tmp)

In [37]:
cmp_df['baseline_accuracy'] = list(map(
    list_exact_match_raw,
    cmp_df.gold_answer,
    cmp_df.baseline_answer))

cmp_df['mipro_accuracy'] = list(map(
    list_exact_match_raw,
    cmp_df.gold_answer,
    cmp_df.mipro_answer))

cmp_df['bfs_rs_accuracy'] = list(map(
    list_exact_match_raw,
    cmp_df.gold_answer,
    cmp_df.bfs_rs_answer))

In [38]:
cmp_df[['baseline_accuracy', 'mipro_accuracy', 'bfs_rs_accuracy']].mean()*100

baseline_accuracy    73.584906
mipro_accuracy       81.132075
bfs_rs_accuracy      79.245283
dtype: float64

In [39]:
bfs_rs_optimized_nps_topic_predictor(
    comment = "Absolutely frustrated! Every time I find something I love, it's sold out in my size."
    "What's the point of having a wishlist if nothing is ever available?"
)

Prediction(
    reasoning='The customer is expressing frustration because items they are interested in are consistently unavailable in their size. This directly relates to the availability of specific sizes and shades for products.',
    answer=['Limited Size or Shade Availability']
)

In [40]:
dspy.inspect_history(n = 1)





[2026-05-13T22:25:35.672264]

System message:

Your input fields are:
1. `comment` (str):
Your output fields are:
1. `reasoning` (str): 
2. `answer` (list[Literal['Unresponsive or Generic Customer Support', 'Difficult Product Discovery', 'Website or App Bugs', 'Customs and Import Charges', 'Confusing Loyalty or Discount Systems', 'Limited Size or Shade Availability', 'Slow or Unreliable Shipping', 'Inaccurate Product Descriptions or Photos', 'Complicated Returns or Exchanges', 'Damaged or Incorrect Items']]):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## comment ## ]]
{comment}

[[ ## reasoning ## ]]
{reasoning}

[[ ## answer ## ]]
{answer}        # note: the value you produce must adhere to the JSON schema: {"type": "array", "items": {"type": "string", "enum": ["Unresponsive or Generic Customer Support", "Difficult Product Discovery", "Website or App Bugs", "Customs and Import Charges", "Confusing Loyalty or Discount System